# HW9 · Diagnose and defend a model change
**Course:** Machine Learning in Engineering Science · NCKU + LAiMM · Fall 2026.
**Format:** individual Homework, approximately two hours. AI assistance is allowed with honest disclosure and your own checks. No paid API, GPU or live data service is required.
**Assessment boundary:** the midterm has written and oral examinations only, with no midterm project. Stock forecasting remains a separate Homework. Submission platform and course-grade percentage are to be announced.

**Due:** 8 November 2026,23:59 Asia/Taipei. Submit the edited executed notebook, the generated evidence ZIP, and the separate one-page English explanation. A successful run with no code change remains a draft. No score improvement is required for a strong explanation.

## Route ·120 minutes
Read and formulate 10; validation diagnosis 25; hypothesis and implementation 30; freeze and test 20; explain and export 25.

Start with a copy in your own Drive. In Colab use a standard Python CPU runtime. In local Jupyter, install NumPy, SciPy, scikit-learn and Matplotlib if imports are missing. The setup embeds the support code and exact fixed CSV, so the learning path does not depend on an external download. The helpers provide routine validation and packaging; you own the modeling decisions.

## 1. Read the data contract before fitting
Each row is one independently simulated equipment experiment. Inputs are applied power in W, airflow in m/s, and ambient temperature in °C. The target is **temperature rise**, a difference in °C, not absolute device temperature. The records are synthetic: their fitted relationships are not validated equipment physics. The supplied split is360 train /120 validation /120 test. Never move IDs between splits. Scaling fits training only. Explain why IID splitting is reasonable for these constructed independent rows and why that does not justify random splitting of a real time series.

In [ ]:
from pathlib import Path
import json,base64,zlib,importlib,sys
import numpy as np
import matplotlib.pyplot as plt
# Support code is embedded so the notebook also runs without repository downloads.
HELPER_SOURCE = '"""Support code for two individual ML Homework assignments; synthetic data only.\nNo network, API key, GPU, pickle or joblib is used. Review submitted Python source\nbefore executing another student\'s package. Integrity hashes are not signatures.\n"""\nfrom pathlib import Path\nimport csv, json, hashlib, io, zipfile, tempfile, shutil, sys, subprocess, platform, html, base64, importlib.util, uuid, contextlib\nimport numpy as np\nimport sklearn, scipy\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.tree import DecisionTreeRegressor\nFEATURES=[\'power_W\',\'airflow_m_s\',\'ambient_C\']\nCOLUMNS=[\'run_id\',*FEATURES,\'temp_rise_C\',\'split\']\nUNITS={\'power_W\':\'W\',\'airflow_m_s\':\'m/s\',\'ambient_C\':\'degC\',\'temp_rise_C\':\'degC difference\'}\nBASE_CONFIG={\'learner\':\'ridge\',\'alpha\':1.0,\'max_depth\':3}\nGROUPS={\'low_airflow\':lambda X:X[:,1]<=1.,\'high_power\':lambda X:X[:,0]>=75.,\'hot_ambient\':lambda X:X[:,2]>=28.}\nREPORT_FIELDS=[\'change\',\'prediction\',\'result\',\'evidence\',\'limitations\']\n\ndef digest(data):return hashlib.sha256(data).hexdigest()\ndef jd(data):return json.dumps(data,sort_keys=True,separators=(\',\',\':\'),allow_nan=False)\ndef versions():return {\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'scikit-learn\':sklearn.__version__}\ndef write_json(path,data):Path(path).write_text(json.dumps(data,indent=2,allow_nan=False),encoding=\'utf8\')\ndef load_table(path,contract):\n    if contract.get(\'units\')!=UNITS:raise ValueError(\'unit contract mismatch; do not silently relabel or convert inputs\')\n    with Path(path).open(newline=\'\',encoding=\'utf8\') as f:\n        reader=csv.DictReader(f)\n        if reader.fieldnames!=COLUMNS:raise ValueError(\'CSV columns or order differ from the declared schema\')\n        rows=list(reader)\n    ids=[r[\'run_id\'] for r in rows]\n    if len(ids)!=len(set(ids)):raise ValueError(\'duplicate run_id\')\n    if len(rows)!=contract[\'rows\']:raise ValueError(\'row count mismatch\')\n    numeric=[]\n    for r in rows:\n        try:v=[float(r[k]) for k in FEATURES+[\'temp_rise_C\']]\n        except (ValueError,TypeError):raise ValueError(\'non-numeric measurement\') from None\n        if not np.all(np.isfinite(v)):raise ValueError(\'non-finite measurement\')\n        if not (0<=v[0]<=200 and .1<=v[1]<=10 and 0<=v[2]<=60):raise ValueError(\'input outside declared encoding bounds\')\n        if r[\'split\'] not in [\'train\',\'validation\',\'test\']:raise ValueError(\'unknown split label\')\n        numeric.append(v)\n    order=np.argsort(ids);rows=[rows[i] for i in order];numeric=np.asarray(numeric)[order];ids=np.array(ids)[order]\n    records=[{\'run_id\':r[\'run_id\'],**{k:float(r[k]) for k in FEATURES+[\'temp_rise_C\']},\'split\':r[\'split\']} for r in rows]\n    if digest(jd(records).encode())!=contract[\'records_sha256\']:raise ValueError(\'ID-keyed records differ from the frozen data contract\')\n    split=np.array([r[\'split\'] for r in rows]);counts={k:int(np.sum(split==k)) for k in [\'train\',\'validation\',\'test\']}\n    if counts!=contract[\'split_counts\']:raise ValueError(\'split counts mismatch\')\n    return {\'ids\':ids,\'X\':numeric[:,:3],\'y\':numeric[:,3],\'split\':split,\'records_sha256\':contract[\'records_sha256\']}\n\ndef features_from_source(source):\n    # This is ordinary reviewed course Python, not a sandbox for hostile code.\n    scope={\'np\':np};exec(compile(source,\'candidate_features.py\',\'exec\'),scope)\n    if not callable(scope.get(\'candidate_features\')):raise ValueError(\'define candidate_features(X)\')\n    return scope[\'candidate_features\']\ndef checked_features(fn,X):\n    before=X.copy();a=np.asarray(fn(X),dtype=float);b=np.asarray(fn(X),dtype=float)\n    if not np.array_equal(before,X):raise ValueError(\'feature function changed X in place\')\n    if a.ndim!=2 or a.shape[0]!=len(X) or a.shape[1]<3:raise ValueError(\'feature shape must be rows by at least three columns\')\n    if not np.array_equal(a[:,:3],X):raise ValueError(\'preserve the three original columns first\')\n    if not np.all(np.isfinite(a)) or not np.array_equal(a,b):raise ValueError(\'features must be finite and deterministic\')\n    return a\n\ndef estimator(config):\n    if set(config)!=set(BASE_CONFIG) or config[\'learner\'] not in [\'ridge\',\'tree\']:raise ValueError(\'unknown model configuration\')\n    alpha=config[\'alpha\'];depth=config[\'max_depth\']\n    if not isinstance(alpha,(int,float)) or isinstance(alpha,bool) or not np.isfinite(alpha) or not (0<alpha<=1000):raise ValueError(\'alpha must be finite in (0,1000]\')\n    if not isinstance(depth,int) or isinstance(depth,bool) or not (1<=depth<=6):raise ValueError(\'tree depth must be an integer from 1 to 6\')\n    learner=Ridge(alpha=alpha,solver=\'svd\') if config[\'learner\']==\'ridge\' else DecisionTreeRegressor(max_depth=depth,min_samples_leaf=15,random_state=2031)\n    return make_pipeline(StandardScaler(),learner)\ndef score(y,p):return {\'MAE\':float(np.mean(np.abs(p-y))),\'RMSE\':float(np.sqrt(np.mean((p-y)**2))),\'n\':len(y)}\ndef comparison(table,source,config,include_test=False):\n    X,y,split=table[\'X\'],table[\'y\'],table[\'split\'];tr=split==\'train\';va=split==\'validation\'\n    fn=features_from_source(source);trans=checked_features(fn,X.copy())\n    if config[\'learner\']==\'ridge\' and config[\'max_depth\']!=BASE_CONFIG[\'max_depth\']:raise ValueError(\'max_depth has no effect on Ridge; change an active setting\')\n    if config[\'learner\']==\'tree\' and config[\'alpha\']!=BASE_CONFIG[\'alpha\']:raise ValueError(\'alpha has no effect on this tree; retain the default unused field\')\n    feature_change=not(trans.shape==X.shape and np.array_equal(trans,X));model_change=config!=BASE_CONFIG\n    if feature_change and model_change:raise ValueError(\'change the feature basis OR model settings, one factor family at a time\')\n    baseline=estimator(BASE_CONFIG).fit(X[tr],y[tr]);candidate=estimator(config).fit(trans[tr],y[tr])\n    result={\'implemented\':feature_change or model_change,\'versions\':versions(),\'validation\':{\'baseline\':score(y[va],baseline.predict(X[va])),\'candidate\':score(y[va],candidate.predict(trans[va]))}}\n    result[\'selected\']=\'candidate\' if result[\'validation\'][\'candidate\'][\'MAE\']<result[\'validation\'][\'baseline\'][\'MAE\'] else \'baseline\'\n    if include_test:\n        te=split==\'test\';pb=baseline.predict(X[te]);pc=candidate.predict(trans[te]);result[\'test\']={\'baseline\':score(y[te],pb),\'candidate\':score(y[te],pc)}\n        result[\'slices\']={}\n        for name,group in GROUPS.items():\n            mask=group(X[te])\n            if mask.any():result[\'slices\'][name]={\'baseline\':score(y[te][mask],pb[mask]),\'candidate\':score(y[te][mask],pc[mask])}\n        result[\'test_ids\']=table[\'ids\'][te].tolist();result[\'predictions\']={\'baseline\':pb.tolist(),\'candidate\':pc.tolist()}\n    return result\n\ndef freeze_plan(path,source,config,plan,table):\n    if plan.get(\'failure_group\') not in GROUPS:raise ValueError(\'choose low_airflow,high_power,or hot_ambient before freezing\')\n    result=comparison(table,source,config)\n    proposal={\'source\':source,\'config\':config,\'plan\':plan,\'records_sha256\':table[\'records_sha256\'],\'selected_on_validation\':result[\'selected\'],\'validation\':result[\'validation\']}\n    proposal[\'proposal_sha256\']=digest(jd(proposal).encode());write_json(path,proposal);return proposal\n\ndef run_frozen(table,frozen,source,config,plan):\n    check={k:v for k,v in frozen.items() if k!=\'proposal_sha256\'}\n    if digest(jd(check).encode())!=frozen[\'proposal_sha256\'] or source!=frozen[\'source\'] or config!=frozen[\'config\'] or plan!=frozen[\'plan\'] or table[\'records_sha256\']!=frozen[\'records_sha256\']:raise ValueError(\'proposal changed after freeze; declare a new plan and disclose any repeated test use\')\n    result=comparison(table,source,config,include_test=True)\n    if result[\'selected\']!=frozen[\'selected_on_validation\']:raise ValueError(\'validation choice differs\')\n    return result\n\ndef one_page(path,title,fields,disclosure):\n    # Five short sections and disclosure; this file is the separately submitted explanation.\n    words=sum(len(str(v).split()) for v in fields.values())+len(str(disclosure).split())\n    if words>450:raise ValueError(\'one-page explanation exceeds 450 words; shorten it\')\n    body=\'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>\'+html.escape(title)+\'</title><style>@page{size:A4;margin:18mm}body{max-width:760px;margin:20px auto;padding:12px;font:11pt/1.4 Arial;color:#12233f}h1{font-size:18pt}h2{font-size:12pt;margin:14px 0 4px}p{margin:0}footer{margin-top:18px;font-size:9pt;color:#52666b}</style><h1>\'+html.escape(title)+\'</h1>\'\n    for key in REPORT_FIELDS:body+=\'<h2>\'+html.escape(key.title())+\'</h2><p>\'+html.escape(str(fields.get(key,\'\')).strip() or \'[Complete this section]\')+\'</p>\'\n    body+=\'<h2>AI disclosure</h2><p>\'+html.escape(str(disclosure) or \'[State assistance or none]\')+\'</p><footer>NCKU + LAiMM · Individual Homework · Synthetic teaching case · \'+str(words)+\' words</footer></html>\'\n    Path(path).write_text(body,encoding=\'utf8\');return words\n\ndef hw9_export(csv_path,contract,frozen,result,fields,disclosure,out_dir=\'hw09-output\'):\n    table=load_table(csv_path,contract);check=run_frozen(table,frozen,frozen[\'source\'],frozen[\'config\'],frozen[\'plan\'])\n    if jd(check)!=jd(result):raise ValueError(\'result does not match the frozen executable protocol\')\n    status=\'ready for review\' if result[\'implemented\'] and all(str(frozen[\'plan\'].get(k,\'\')).strip() for k in [\'hypothesis\',\'expected\',\'held_fixed\',\'failure_condition\']) and all(str(fields.get(k,\'\')).strip() for k in REPORT_FIELDS) and str(disclosure).strip() else \'draft\'\n    target=Path(out_dir);target.mkdir(exist_ok=True);run=target/(\'run-\'+uuid.uuid4().hex[:10]);run.mkdir();bundle=run/\'bundle\';bundle.mkdir();figdir=bundle/\'figures\';figdir.mkdir()\n    shutil.copy2(csv_path,bundle/\'frozen.csv\');write_json(bundle/\'contract.json\',contract);write_json(bundle/\'experiment.json\',frozen);write_json(bundle/\'result.json\',result);write_json(bundle/\'environment.json\',versions());(bundle/\'candidate_features.py\').write_text(frozen[\'source\']);shutil.copy2(Path(__file__),bundle/\'hw_workflow.py\')\n    with (bundle/\'reference-predictions.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.writer(f);writer.writerow([\'run_id\',\'baseline\',\'candidate\'])\n        writer.writerows(zip(result[\'test_ids\'],result[\'predictions\'][\'baseline\'],result[\'predictions\'][\'candidate\']))\n    replay="from pathlib import Path\\nimport json,sys\\nfrom hw_workflow import replay_directory\\nif __name__==\'__main__\':\\n    result=replay_directory(Path(__file__).parent)\\n    print(json.dumps(result,indent=2))\\n"\n    (bundle/\'replay.py\').write_text(replay)\n    (bundle/\'requirements.txt\').write_text(\'\\n\'.join(k+\'==\'+v for k,v in versions().items() if k!=\'python\')+\'\\n\')\n    (bundle/\'README.txt\').write_text(\'Inspect the Python source first. Use CPU and the recorded package versions. Run: python replay.py\\nThis is a reviewed class package, not a sandbox. A new process is not automatically an independent person or environment.\\nThe receiver compares every ID-aligned baseline and candidate test prediction with atol=1e-8,rtol=1e-10. Do not change the frozen selection from test scores.\\n\')\n    import matplotlib.pyplot as plt\n    X,y,split=table[\'X\'],table[\'y\'],table[\'split\'];te=split==\'test\'\n    fig,axes=plt.subplots(1,2,figsize=(10,4),sharex=True,sharey=True)\n    for ax,kind in zip(axes,[\'baseline\',\'candidate\']):ax.scatter(X[te,1],np.asarray(result[\'predictions\'][kind])-y[te],s=20);ax.axhline(0,c=\'black\',lw=1);ax.set(title=kind,xlabel=\'airflow (m/s)\',ylabel=\'prediction - observation (degC)\')\n    fig.tight_layout();fig.savefig(figdir/\'residuals.png\',dpi=140);plt.show();plt.close(fig)\n    fig,ax=plt.subplots(figsize=(7,4));ax.bar([\'baseline\',\'candidate\'],[result[\'test\'][k][\'MAE\'] for k in [\'baseline\',\'candidate\']]);ax.set(ylabel=\'test MAE (degC)\',title=\'Predeclared diagnosis; selection was frozen on validation\');fig.tight_layout();fig.savefig(figdir/\'comparison.png\',dpi=140);plt.show();plt.close(fig)\n    words=one_page(run/\'HW9-explanation.html\',\'HW9 · Diagnose and defend a change\',fields,disclosure);shutil.copy2(run/\'HW9-explanation.html\',bundle/\'explanation.html\')\n    gallery=\'<!doctype html><html lang="en"><meta charset="utf-8"><title>HW9 evidence</title><style>body{max-width:1000px;margin:auto;font:18px/1.5 sans-serif}img{max-width:100%}pre{white-space:pre-wrap}</style><h1>HW9 evidence: \'+status+\'</h1><pre>\'+html.escape(json.dumps({\'validation\':result[\'validation\'],\'test\':result[\'test\'],\'slices\':result[\'slices\']},indent=2))+\'</pre>\'\n    for file in sorted(figdir.glob(\'*.png\')):gallery+=\'<h2>\'+file.stem+\'</h2><img alt="\'+file.stem+\'" src="data:image/png;base64,\'+base64.b64encode(file.read_bytes()).decode()+\'">\'\n    (bundle/\'evidence-report.html\').write_text(gallery)\n    manifest={\'assignment\':\'HW9\',\'status\':status,\'scope\':\'Synthetic individual experiment; presence is not correctness.\',\'files\':{str(p.relative_to(bundle)):digest(p.read_bytes()) for p in sorted(bundle.rglob(\'*\')) if p.is_file()}}\n    write_json(bundle/\'manifest.json\',manifest)\n    package=run/\'hw09-evidence.zip\'\n    with zipfile.ZipFile(package,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(bundle.rglob(\'*\')):\n            if p.is_file():z.write(p,str(p.relative_to(bundle)))\n    print(\'HW9:\',status,\'\\nEvidence ZIP:\',package,\'\\nSeparate one-page explanation:\',run/\'HW9-explanation.html\',\'\\nWords:\',words)\n    return {\'status\':status,\'zip\':str(package.resolve()),\'explanation\':str((run/\'HW9-explanation.html\').resolve()),\'bundle\':str(bundle.resolve())}\n\ndef validate_manifest(folder):\n    folder=Path(folder);m=json.loads((folder/\'manifest.json\').read_text())\n    for name,h in m[\'files\'].items():\n        p=Path(name)\n        if p.is_absolute() or \'..\' in p.parts:raise ValueError(\'unsafe manifest path\')\n        if not (folder/name).is_file() or digest((folder/name).read_bytes())!=h:raise ValueError(\'file checksum mismatch: \'+name)\n    return m\n\ndef unpack(package,destination):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=False)\n    with zipfile.ZipFile(package) as z:\n        if sum(i.file_size for i in z.infolist())>20_000_000:raise ValueError(\'class package exceeds 20 MB\')\n        for i in z.infolist():\n            p=Path(i.filename)\n            if p.is_absolute() or \'..\' in p.parts or ((i.external_attr>>16)&0o170000)==0o120000:raise ValueError(\'unsafe ZIP path or symlink\')\n        z.extractall(destination)\n    validate_manifest(destination);return destination\n\ndef replay_directory(folder):\n    folder=Path(folder);m=validate_manifest(folder)\n    if m[\'status\']!=\'ready for review\':raise ValueError(\'HW9 package is a draft; finish your actual change and explanation first\')\n    contract=json.loads((folder/\'contract.json\').read_text());table=load_table(folder/\'frozen.csv\',contract);frozen=json.loads((folder/\'experiment.json\').read_text())\n    if (folder/\'candidate_features.py\').read_text()!=frozen[\'source\']:raise ValueError(\'feature source differs from frozen declaration\')\n    result=run_frozen(table,frozen,frozen[\'source\'],frozen[\'config\'],frozen[\'plan\'])\n    with (folder/\'reference-predictions.csv\').open() as f:records=list(csv.DictReader(f))\n    ids=[r[\'run_id\'] for r in records]\n    if len(ids)!=len(set(ids)) or set(ids)!=set(result[\'test_ids\']):raise ValueError(\'reference prediction IDs mismatch\')\n    by_id={r[\'run_id\']:r for r in records};differences={}\n    for name in [\'baseline\',\'candidate\']:\n        expected=np.array([float(by_id[i][name]) for i in result[\'test_ids\']]);actual=np.array(result[\'predictions\'][name])\n        if not np.allclose(actual,expected,rtol=1e-10,atol=1e-8):raise ValueError(\'prediction vector mismatch for \'+name)\n        differences[name]=float(np.max(np.abs(actual-expected)))\n    return {\'status\':\'reproduced\',\'rows_compared\':len(ids),\'max_abs_difference\':differences,\'atol\':1e-8,\'rtol\':1e-10,\'selected_on_validation\':result[\'selected\'],\'receiver_versions\':versions(),\'sender_versions\':json.loads((folder/\'environment.json\').read_text()),\'scope\':\'New Python process in receiver working directory; independence requires a separate human receiver.\'}\n\ndef fresh_process_replay(package,destination):\n    folder=unpack(package,destination)\n    process=subprocess.run([sys.executable,str((folder/\'replay.py\').resolve())],cwd=folder,capture_output=True,text=True,timeout=60)\n    receipt={\'returncode\':process.returncode,\'stdout\':process.stdout,\'stderr\':process.stderr,\'command\':\'python replay.py\',\'fresh_process\':True}\n    if process.returncode==0:receipt[\'result\']=json.loads(process.stdout)\n    return folder,receipt\n\ndef negative_test(folder,mutate_contract):\n    contract=json.loads((Path(folder)/\'contract.json\').read_text());before=jd(contract);bad=json.loads(before);mutate_contract(bad)\n    if jd(bad)==before:return {\'status\':\'not implemented\',\'expected_rejection\':False}\n    try:load_table(Path(folder)/\'frozen.csv\',bad)\n    except ValueError as e:return {\'status\':\'rejected\',\'expected_rejection\':True,\'exception\':type(e).__name__,\'message\':str(e)}\n    return {\'status\':\'unexpectedly accepted\',\'expected_rejection\':False}\n\ndef hw10_export(receipt,rejection,fields,disclosure,verification_mode,receiver_status,history=None,package_path=None,out_dir=\'hw10-output\'):\n    if verification_mode not in [\'peer\',\'self-replay\']:raise ValueError(\'mode must honestly be peer or self-replay\')\n    if verification_mode==\'peer\' and not str(receiver_status).strip():raise ValueError(\'describe the actual independent receiver and action; a mode label is not evidence\')\n    complete=receipt.get(\'returncode\')==0 and rejection.get(\'expected_rejection\') and all(str(fields.get(k,\'\')).strip() for k in REPORT_FIELDS) and str(disclosure).strip() and str(receiver_status).strip()\n    status=(\'ready for review\' if verification_mode==\'peer\' else \'self-replay; instructor/TA verification pending\') if complete else \'draft or unresolved replay failure\'\n    out=Path(out_dir)/(\'run-\'+uuid.uuid4().hex[:10]);out.mkdir(parents=True);write_json(out/\'replay-receipt.json\',receipt);write_json(out/\'negative-test.json\',rejection);write_json(out/\'verification.json\',{\'mode\':verification_mode,\'receiver_status\':receiver_status,\'status\':status,\'AI_disclosure\':disclosure,\'presence_is_not_correctness\':True});write_json(out/\'receiver-environment.json\',versions());one_page(out/\'HW10-explanation.html\',\'HW10 · Handover and reproducibility\',fields,disclosure)\n    write_json(out/\'replay-history.json\',history or [receipt])\n    if package_path is not None and Path(package_path).is_file():write_json(out/\'package-identity.json\',{\'filename\':Path(package_path).name,\'sha256\':digest(Path(package_path).read_bytes())})\n    files=sorted(p for p in out.iterdir() if p.is_file());write_json(out/\'manifest.json\',{\'status\':status,\'files\':{p.name:digest(p.read_bytes()) for p in files}})\n    package=out/\'hw10-evidence.zip\'\n    with zipfile.ZipFile(package,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.iterdir()):\n            if p.is_file() and p!=package:z.write(p,p.name)\n    print(\'HW10:\',status,\'\\nEvidence ZIP:\',package,\'\\nSeparate one-page explanation:\',out/\'HW10-explanation.html\')\n    return {\'status\':status,\'zip\':str(package.resolve()),\'explanation\':str((out/\'HW10-explanation.html\').resolve())}\n'
Path('hw_workflow.py').write_text(HELPER_SOURCE,encoding='utf8')
import hw_workflow as hw
importlib.reload(hw)
print('Runtime:',hw.versions())

In [ ]:
DATA_BYTES=zlib.decompress(base64.b64decode('...'))
Path('thermal-sensor.csv').write_bytes(DATA_BYTES)
CONTRACT={'case': 'Independent synthetic equipment temperature-rise experiments', 'rows': 600, 'split_counts': {'train': 360, 'validation': 120, 'test': 120}, 'units': {'power_W': 'W', 'airflow_m_s': 'm/s', 'ambient_C': 'degC', 'temp_rise_C': 'degC difference'}, 'records_sha256': '3d197032b92337a88dc630e5313d2be65eef776a35118adec61130b495daaecf', 'source': 'Original course-generated synthetic fixture; no measured plant data or physical validation.', 'split_rationale': 'Independent simulated runs,not a time series. Fixed disjoint row IDs; random IID split is a teaching assumption.', 'encoding_bounds': {'power_W': [0, 200], 'airflow_m_s': [0.1, 10], 'ambient_C': [0, 60]}, 'target': 'Temperature rise above the ambient condition,not absolute temperature.'}
TABLE=hw.load_table('thermal-sensor.csv',CONTRACT)
print('Frozen data:',TABLE['records_sha256'],CONTRACT['split_counts'])

## 2. Diagnose the preserved baseline using validation only
The baseline is train-fitted StandardScaler → Ridge(alpha=1, SVD). Read residual sign and units. The three group rules are declared in advance: low_airflow ≤ 1 m/s, high_power ≥ 75 W, hot_ambient ≥ 28°C. Use a group with enough observations; always report its count. Choose one failure condition from the validation evidence. Do not open test metrics to discover or optimize your proposal.

In [ ]:
train=TABLE['split']=='train';val=TABLE['split']=='validation'
base=hw.estimator(hw.BASE_CONFIG).fit(TABLE['X'][train],TABLE['y'][train])
vp=base.predict(TABLE['X'][val]);residual=vp-TABLE['y'][val]
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,j,label in zip(axes,[0,1],['power (W)','airflow (m/s)']):
    ax.scatter(TABLE['X'][val,j],residual,s=22);ax.axhline(0,c='black',lw=1);ax.set(xlabel=label,ylabel='prediction - observation (degC)')
plt.tight_layout();plt.show();plt.close(fig)
for name,fn in hw.GROUPS.items():
    mask=fn(TABLE['X'][val]);print(name,'count',int(mask.sum()),hw.score(TABLE['y'][val][mask],vp[mask]))
print('Use validation evidence to choose a failure group; test outcomes have not been computed here.')

## 3. Predict, then edit one modeling assumption
Write a precise hypothesis and expected direction before fitting your change. Preserve all original columns. A derived feature must use X only and return a finite deterministic matrix without mutating its input. A ratio or interaction may express a relationship an affine raw-input model cannot; explain its units and limits.

Alternatively keep the identity features and change one active model setting: Ridge alpha or a small decision tree (`learner="tree"`, depth1–6, minimum leaf15). Keep unused fields at their defaults. Do not change both feature basis and model settings. A comment, seed-only edit or unchanged identity does not satisfy the task.

The support cell checks validation MAE and freezes the selection between the original baseline and your one proposed candidate; ties preserve baseline. Report both candidates for the predeclared test diagnosis without reselecting on test outcomes.

In [ ]:
# Choose ONE X-only feature edit OR one model-setting change.
FEATURE_SOURCE = '''def candidate_features(X):
    return X.copy()
'''
CONFIG={'learner':'ridge','alpha':1.0,'max_depth':3}
PLAN={
 'hypothesis':'',
 'expected':'',
 'held_fixed':'',
 'failure_group':'low_airflow', # choose from the three declared validation slices
 'failure_condition':''
}

In [ ]:
VALIDATION=hw.comparison(TABLE,FEATURE_SOURCE,CONFIG)
print(json.dumps(VALIDATION,indent=2))

## 4. Freeze before test
Once the hypothesis, expected outcome, held-fixed conditions and selected failure group are complete, run Freeze. Then run the next cell once to expose the predeclared test comparison. The digest records source/config/plan identity, not honesty or correctness. If you revise after seeing test outcomes, clearly label the repeated use exploratory. Do not claim a new untouched test set.

In [ ]:
FROZEN=hw.freeze_plan('frozen-proposal.json',FEATURE_SOURCE,CONFIG,PLAN,TABLE)
print('Frozen:',FROZEN['proposal_sha256'],'Selected on validation:',FROZEN['selected_on_validation'])

In [ ]:
RESULT=hw.run_frozen(TABLE,FROZEN,FEATURE_SOURCE,CONFIG,PLAN)
print(json.dumps({k:RESULT[k] for k in ['validation','test','slices','selected','implemented']},indent=2))

## 5. Write your one-page English explanation
Complete the five fields below, using 300–450 words in total including AI disclosure. **Change:** target, split, baseline, and what you changed. **Prediction:** what you expected and what stayed fixed. **Result:** actual baseline/candidate metrics and one slice count. **Evidence:** point to code, plots and the frozen declaration. **Limitations:** what this experiment cannot establish and one controlled next test. State assistance or none; explain how you checked generated code. Completeness is not correctness. An honest unsuccessful candidate can receive high marks.

In [ ]:
REPORT={
 'change':'',
 'prediction':'',
 'result':'',
 'evidence':'',
 'limitations':''
}
AI_DISCLOSURE='' # Describe the actual assistance and checks,or state none.

## 6. Export and inspect
Run the cell. It creates a fresh run directory, so previous figures cannot leak into your ZIP. Open `evidence-report.html` for the actual metrics and two charts. The separate `HW9-explanation.html` is the one-page submission: open it locally and optionally print to PDF after checking one-page layout. In Colab use the Files sidebar, expand hw09-output/run-…, right-click the ZIP and explanation to Download; save the edited notebook separately with File → Download → .ipynb.

The ZIP also contains the source, data contract, exact CSV, ID-aligned reference predictions, environment and `replay.py`. Keep it for HW10. A draft still exports for debugging, but you must implement your change and explanation before claiming completion. Colab rich-output JavaScript problems do not establish Python failure: save, reload, and inspect exported PNGs/offline HTML. No browser setting change is required by these instructions.

In [ ]:
EXPORT=hw.hw9_export('thermal-sensor.csv',CONTRACT,FROZEN,RESULT,REPORT,AI_DISCLOSURE)
print(EXPORT)

## Rubric ·100 points
Problem and evaluation design20; controlled code modification30; error analysis and limitations30; reproducible evidence and AI disclosure20. Accuracy alone does not determine the grade. Course-grade percentage and submission platform: TBA.

## Self-check
Can you independently defend target, split, baseline, evaluation and limitations? Can you locate the actual changed source and frozen hypothesis? Are both model scores and group counts reported? Are all three submission files saved?